In [ ]:
from pathlib import Path
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
CODEBASE_DIR = Path('/content/drive/MyDrive/predicting-the-future-effect-of-optimizer-memory')  
assert (CODEBASE_DIR / 'requirements.txt').is_file(), 'Upload and unzip the code folder to Drive first.'
os.chdir(CODEBASE_DIR)
sys.path.insert(0, str(CODEBASE_DIR))
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])


In [ ]:
from rttp_grid.amortization import AmortizationConfig

ROOT_NAME = 'results/rttp_amortization_cifar'
RUN_NAME = 'rttp_amortization_cifar'

ARCHITECTURE = 'smallcnn'  
SEED = 0
HORIZON = 40
RESPONSE_RANK = 8
CANDIDATE_COUNTS = (5, 10, 20, 50)
REPEATS = 3
WARMUP = 1


LEARNING_RATE = 0.03
MOMENTUM = 0.9
RADIUS_SCALE = 1e-3


FORCE_BOUNDARY = False

cfg = AmortizationConfig(
    run_name=RUN_NAME,
    root_name=ROOT_NAME,
    architecture=ARCHITECTURE,
    seed=SEED,
    horizon=HORIZON,
    learning_rate=LEARNING_RATE,
    momentum=MOMENTUM,
    radius_scale=RADIUS_SCALE,
    response_rank=RESPONSE_RANK,
    candidate_counts=CANDIDATE_COUNTS,
    repeats=REPEATS,
    warmup=WARMUP,
    force_boundary=FORCE_BOUNDARY,
)

cfg

In [ ]:
from rttp_grid.amortization import run_amortization_experiment

output_root = run_amortization_experiment(cfg)
print('output_root:', output_root)

In [ ]:
import pandas as pd
from pathlib import Path

from rttp_grid.utils import get_root
root = get_root(cfg)
summary = pd.read_csv(root / 'tables' / 'amortization_summary.csv')
raw = pd.read_csv(root / 'tables' / 'amortization_timing_raw.csv')
quality = pd.read_csv(root / 'tables' / 'amortization_prediction_quality.csv')
env = pd.read_json(root / 'environment.json', typ='series')

display(summary)
display(quality.groupby('candidate_count').mean(numeric_only=True).reset_index())
display(raw.groupby(['phase', 'candidate_count'])[['seconds', 'peak_memory_mb']].mean().reset_index())
print(env.to_string())

In [ ]:
latex_path = root / 'tables' / 'amortization_table.tex'
print(latex_path.read_text())

In [ ]:
import shutil

archive = shutil.make_archive(str(root), 'zip', root)
print('archive:', archive)